In [2]:
"""
Final Merge: Undersampled Normal + Real Attacks + GAN-Generated Synthetic Samples
====================================================================================
Assembles the final balanced training set for handoff to Keerthi's classifier
pipeline. Combines:
  1. Undersampled Normal class (from knn_undersample_normal.py output)
  2. Real attack samples (DoS, Probe, R2L, U2R -- untouched)
  3. Freshly generated synthetic samples to bring each attack class up toward
     TARGET_COUNT:
       - U2R  -> generated by ACGAN (best mean_abs_diff for this class)
       - R2L, Probe, DoS -> generated by WCGAN (best mean_abs_diff for these)

IMPORTANT DESIGN NOTE ON TARGET COUNTS:
Full balancing (bringing every class to 42401) means very high oversampling
ratios for the smallest classes:
  - U2R: 198 real -> would need 42203 synthetic (~213x oversampling ratio)
  - R2L: 2985 real -> would need 39416 synthetic (~13x oversampling ratio)
This is aggressive -- a GAN trained on only 198 real U2R samples generating
42000+ synthetic samples risks producing repetitive/low-diversity data even
if individual samples look statistically reasonable (see mode collapse
diagnostic from earlier). This script defaults U2R to a MORE CONSERVATIVE
target (5000) rather than full 42401 -- edit TARGET_COUNTS below if you and
your guide decide differently. This is a judgment call worth documenting
in your methodology chapter either way.

Requires the saved generator weights from wcgan_baseline.py and
acgan_baseline.py (generator_*.pt files in their respective output folders).

Author: Vibha J | GAN Implementation Lead
====================================================================================
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ============================================================
# CONFIG -- EDIT THESE PATHS TO MATCH YOUR LOCAL SETUP
# ============================================================
UNDERSAMPLED_X_PATH = "./data/X_train_undersampled_v2.csv"
UNDERSAMPLED_Y_PATH = "./data/y_train_multiclass_undersampled_v2.csv"

WCGAN_DIR = "./wcgan_outputs"
ACGAN_DIR = "./acgan_outputs"
OUTPUT_DIR = "./final_augmented"

LATENT_DIM = 32
SEED = 42

# Label encoding (from attack_category_encoded): 0=DoS 1=Probe 2=R2L 3=U2R 4=Normal
LABEL_MAP = {"DoS": 0, "Probe": 1, "R2L": 2, "U2R": 3}
REAL_COUNTS = {"DoS": 42401, "Probe": 11168, "R2L": 2985, "U2R": 198}

# Which architecture generates each class's synthetic samples
ARCHITECTURE_PER_CLASS = {
    "DoS": "WCGAN",
    "Probe": "WCGAN",
    "R2L": "WCGAN",
    "U2R": "ACGAN",
}

# Target total count (real + synthetic) per class after augmentation.
# EDIT THIS if you want different balancing ratios.
TARGET_COUNTS = {
    "DoS": 42401,    # already at target, 0 synthetic needed
    "Probe": 42401,
    "R2L": 42401,
    "U2R": 5000,      # conservative cap, NOT full 42401 -- see note above
}

# ACGAN class index order used during its training (must match acgan_baseline.py)
ACGAN_CLASS_NAMES = ["U2R", "R2L", "Probe", "DoS"]

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# MODEL DEFINITIONS (must match training scripts exactly to load weights)
# ============================================================
class WCGANGenerator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Linear(128, output_dim), nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class ACGANGenerator(nn.Module):
    def __init__(self, latent_dim, num_classes, output_dim):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 64), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Linear(128, output_dim), nn.Tanh(),
        )

    def forward(self, z, labels):
        label_input = self.label_emb(labels)
        return self.net(torch.cat([z, label_input], dim=1))


# ============================================================
# ONE-HOT / BINARY COLUMN CLEANUP
# ============================================================
def clean_generated_columns(df):
    """Generator output is continuous (e.g. 0.34) even for columns that
    should be strictly 0/1 (one-hot categorical dummies, binary flags).
    This snaps them back to clean binary/one-hot values post-generation."""
    df = df.copy()

    onehot_groups = {
        "protocol_type": [c for c in df.columns if c.startswith("protocol_type_")],
        "service": [c for c in df.columns if c.startswith("service_")],
        "flag": [c for c in df.columns if c.startswith("flag_")],
    }
    for group_name, cols in onehot_groups.items():
        if not cols:
            continue
        values = df[cols].values
        argmax_idx = np.argmax(values, axis=1)
        onehot = np.zeros_like(values)
        onehot[np.arange(len(values)), argmax_idx] = 1
        df[cols] = onehot

    binary_flags = ["logged_in", "root_shell", "su_attempted", "is_guest_login"]
    for col in binary_flags:
        if col in df.columns:
            df[col] = (df[col] > 0).astype(float)

    return df


# ============================================================
# GENERATE SYNTHETIC SAMPLES FOR ONE CLASS
# ============================================================
def generate_synthetic(class_name, n_needed, feature_cols):
    if n_needed <= 0:
        return None

    architecture = ARCHITECTURE_PER_CLASS[class_name]
    output_dim = len(feature_cols)

    if architecture == "WCGAN":
        model_path = os.path.join(WCGAN_DIR, f"generator_{class_name}.pt")
        G = WCGANGenerator(LATENT_DIM, output_dim).to(device)
        G.load_state_dict(torch.load(model_path, map_location=device))
        G.eval()
        with torch.no_grad():
            z = torch.randn(n_needed, LATENT_DIM, device=device)
            synthetic = G(z).cpu().numpy()

    elif architecture == "ACGAN":
        model_path = os.path.join(ACGAN_DIR, "generator_acgan.pt")
        G = ACGANGenerator(LATENT_DIM, len(ACGAN_CLASS_NAMES), output_dim).to(device)
        G.load_state_dict(torch.load(model_path, map_location=device))
        G.eval()
        class_idx = ACGAN_CLASS_NAMES.index(class_name)
        with torch.no_grad():
            z = torch.randn(n_needed, LATENT_DIM, device=device)
            labels = torch.full((n_needed,), class_idx, dtype=torch.long, device=device)
            synthetic = G(z, labels).cpu().numpy()

    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    synth_df = pd.DataFrame(synthetic, columns=feature_cols)
    synth_df = clean_generated_columns(synth_df)
    print(f"  Generated {n_needed} synthetic {class_name} samples using {architecture}")
    return synth_df


# ============================================================
# MAIN
# ============================================================
def main():
    print("Loading undersampled Normal + real attack data...")
    X_base = pd.read_csv(UNDERSAMPLED_X_PATH)
    y_base = pd.read_csv(UNDERSAMPLED_Y_PATH)
    label_col = y_base.columns[0]
    feature_cols = X_base.columns.tolist()

    print(f"Base dataset: {len(X_base)} rows")
    print(f"Base class distribution:\n{y_base[label_col].value_counts().sort_index()}\n")

    all_X = [X_base]
    all_y = [y_base[label_col]]

    print("Generating synthetic samples per class...")
    for class_name, label_val in LABEL_MAP.items():
        target = TARGET_COUNTS[class_name]
        real_count = REAL_COUNTS[class_name]
        n_needed = max(0, target - real_count)

        ratio = n_needed / real_count if real_count > 0 else float("inf")
        print(f"\n{class_name}: real={real_count}  target={target}  "
              f"synthetic_needed={n_needed}  (oversampling ratio: {ratio:.1f}x)")

        if ratio > 10:
            print(f"  NOTE: high oversampling ratio for {class_name} -- "
                  f"consider documenting this as a limitation.")

        synth_df = generate_synthetic(class_name, n_needed, feature_cols)
        if synth_df is not None:
            all_X.append(synth_df)
            all_y.append(pd.Series([label_val] * n_needed))

    # ---- Combine everything ----
    X_final = pd.concat(all_X, axis=0, ignore_index=True)
    y_final = pd.concat(all_y, axis=0, ignore_index=True)

    # ---- Shuffle ----
    shuffle_idx = np.random.RandomState(SEED).permutation(len(X_final))
    X_final = X_final.iloc[shuffle_idx].reset_index(drop=True)
    y_final = y_final.iloc[shuffle_idx].reset_index(drop=True)

    print(f"\n{'='*60}\nFINAL AUGMENTED DATASET\n{'='*60}")
    print(f"Total rows: {len(X_final)}")
    print(f"Final class distribution:")
    print(y_final.value_counts().sort_index())

    X_out_path = os.path.join(OUTPUT_DIR, "X_train_augmented.csv")
    y_out_path = os.path.join(OUTPUT_DIR, "y_train_multiclass_augmented.csv")
    X_final.to_csv(X_out_path, index=False)
    y_final.to_frame(name=label_col).to_csv(y_out_path, index=False)

    print(f"\nSaved -> {X_out_path}")
    print(f"Saved -> {y_out_path}")

    # ---- Summary table for documentation / handoff to Keerthi ----
    summary_rows = []
    for class_name, label_val in LABEL_MAP.items():
        real_count = REAL_COUNTS[class_name]
        target = TARGET_COUNTS[class_name]
        synthetic_added = max(0, target - real_count)
        summary_rows.append({
            "class": class_name,
            "real_samples": real_count,
            "synthetic_samples": synthetic_added,
            "architecture_used": ARCHITECTURE_PER_CLASS[class_name],
            "final_total": real_count + synthetic_added,
        })
    normal_count = int((y_base[label_col] == 4).sum())
    summary_rows.append({
        "class": "Normal",
        "real_samples": normal_count,
        "synthetic_samples": 0,
        "architecture_used": "N/A (KNN undersampled)",
        "final_total": normal_count,
    })
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, "augmentation_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nHandoff summary (for Keerthi / documentation) -> {summary_path}")
    print(summary_df.to_string(index=False))

    print("\nREMINDER: Do NOT touch X_test.csv / y_test_*.csv with any of this.")
    print("Test set must remain 100% real, untouched, unmodified for valid evaluation.")


if __name__ == "__main__":
    main()

Loading undersampled Normal + real attack data...
Base dataset: 99153 rows
Base class distribution:
attack_category_encoded
0    42401
1    11168
2     2985
3      198
4    42401
Name: count, dtype: int64

Generating synthetic samples per class...

DoS: real=42401  target=42401  synthetic_needed=0  (oversampling ratio: 0.0x)

Probe: real=11168  target=42401  synthetic_needed=31233  (oversampling ratio: 2.8x)
  Generated 31233 synthetic Probe samples using WCGAN

R2L: real=2985  target=42401  synthetic_needed=39416  (oversampling ratio: 13.2x)
  NOTE: high oversampling ratio for R2L -- consider documenting this as a limitation.
  Generated 39416 synthetic R2L samples using WCGAN

U2R: real=198  target=5000  synthetic_needed=4802  (oversampling ratio: 24.3x)
  NOTE: high oversampling ratio for U2R -- consider documenting this as a limitation.
  Generated 4802 synthetic U2R samples using ACGAN

FINAL AUGMENTED DATASET
Total rows: 174604
Final class distribution:
0    42401
1    42401
2   